# Persistent junction capture

Ordinary contacts are transient. A junction exists only when a recipe
explicitly promotes an eligible contact, typically after the geometry
is substantially relaxed. Anchors use material coordinates so they
survive adaptive remeshing.

In [ ]:
# Junction angles are expressed in radians.
import math
import tangle
from tangle.units import mm, um

## Every `JunctionPolicy` field

| Setting | Meaning | Choices / units |
| --- | --- | --- |
| `name` | Human-readable capture-policy name. | nonempty string |
| `law_name` | Symbolic downstream junction law. | nonempty string |
| `parameter_set` | Junction-law parameter-table identifier. | integer |
| `max_surface_gap` | Largest surface gap eligible for capture. | m |
| `min_crossing_angle` | Smallest accepted unsigned crossing angle. | rad, 0 to pi/2 |
| `max_crossing_angle` | Largest accepted unsigned crossing angle. | rad, 0 to pi/2 |
| `probability` | Deterministic seeded thinning probability. | 0–1 |
| `seed` | Seed for probabilistic capture. | integer |
| `material_pairs` | Optional allowed material-name pairs. | list of pairs |
| `max_per_fiber_pair` | Maximum anchors captured between one fiber pair. | positive count |
| `min_anchor_separation` | Required material-coordinate separation between anchors. | m |
| `candidate_capacity` | Device candidate-buffer capacity. | positive count |

In [ ]:
# Read defaults from the compiled extension instead of duplicating
# them in documentation that could become stale.
policy = tangle.JunctionPolicy()
fields = ['name', 'law_name', 'parameter_set', 'max_surface_gap', 'min_crossing_angle', 'max_crossing_angle', 'probability', 'seed', 'material_pairs', 'max_per_fiber_pair', 'min_anchor_separation', 'candidate_capacity']
{name: getattr(policy, name) for name in fields}

In [ ]:
# The law name and parameter set are downstream labels; capture filters
# decide which current contacts receive persistent material anchors.
policy = tangle.JunctionPolicy(
    "cured binder contacts",
    "cohesive bond",
    parameter_set=2,
    max_surface_gap=0.2 * um,
    min_crossing_angle=math.radians(20),
    max_crossing_angle=math.pi / 2,
    # Probability is deterministic for a fixed seed and candidate set.
    probability=0.25,
    seed=9,
    material_pairs=[("large", "small"), ("large", "large")],
    max_per_fiber_pair=1,
    min_anchor_separation=50 * um,
    candidate_capacity=100_000,
)
# replace() derives a variant without repeating every filter.
every_contact = policy.replace(name="all binder contacts", probability=1.0)

`capture_junctions(policy)` samples once at that recipe point.
`relax_and_capture(iterations=, capture_every=, policy=)` samples
repeatedly at an explicit cadence independent of scheduler batch size.
Captured junctions are topology/export data; current relaxation does
not enforce their mechanics, so late capture is normally the physically
appropriate workflow.

In [ ]:
recipe = tangle.Recipe(tangle.Cell([1 * mm, 1 * mm, 1 * mm]))
# Late capture records bonds after geometry has settled; contacts before
# this explicit operation remain transient.
recipe.relax_until_converged(max_iterations=5_000)
recipe.capture_junctions(policy)

# Alternative: capture repeatedly while relaxing.
repeated = tangle.Recipe(tangle.Cell([1 * mm, 1 * mm, 1 * mm]))
repeated.relax_and_capture(iterations=2_000, capture_every=250, policy=every_contact)
recipe.operations() + repeated.operations()